In [260]:
import torch
from torch import nn
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# -- DATA ORGANIZATION -- 

findata = pd.read_csv('data/findata.csv')

feature_cols = ['qoq_rev', 'yoy_rev', 'qoq_eps', 'yoy_eps', 'momentum',
                'net_margin', 'gross_margin', 'oper_margin', 'roe',
                'de_ratio', 'current_ratio', 'cash_ratio', 'asset_turn']

# Convert fiscal_period (e.g., "Q1 2021") to sortable format
def parse_quarter(q_str):
    """Converts 'Q1 2021' to a sortable tuple (2021, 1)"""
    parts = q_str.split()
    quarter = int(parts[0][1])
    year = int(parts[1])
    return (year, quarter)

findata['sort_key'] = findata['fiscal_period'].apply(parse_quarter)
findata = findata.sort_values(['ticker', 'sort_key'])

def create_sequences(df, seq_length=7):
    X_sequences = []
    y_labels = []
    target_periods = []
    
    for ticker in df['ticker'].unique():
        ticker_data = df[df['ticker'] == ticker].reset_index(drop=True)
        
        for i in range(len(ticker_data) - seq_length):
            sequence = ticker_data.iloc[i:i+seq_length][feature_cols].values
            
            label = ticker_data.iloc[i+seq_length]['eps_status']
            target_period = ticker_data.iloc[i+seq_length]['fiscal_period']
            
            X_sequences.append(sequence)
            y_labels.append(label)
            target_periods.append(target_period)
    
    return np.array(X_sequences), np.array(y_labels), target_periods

X_all, y_all, periods_all = create_sequences(findata, seq_length=4)

def is_before_q3_2023(period):
    year, quarter = parse_quarter(period)
    return year < 2023 or (year == 2023 and quarter < 3)

def is_q3_q4_2023(period):
    year, quarter = parse_quarter(period)
    return year == 2023 and quarter >= 3

def is_2024(period):
    year, quarter = parse_quarter(period)
    return year >= 2024

train_mask = [is_before_q3_2023(p) for p in periods_all]
val_mask = [is_q3_q4_2023(p) for p in periods_all]
test_mask = [is_2024(p) for p in periods_all]

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

In [262]:
X_train = torch.from_numpy(X_train).to(torch.float32)
X_val = torch.from_numpy(X_val).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)

y_train = torch.from_numpy(y_train).to(torch.float32)
y_val = torch.from_numpy(y_val).to(torch.float32)
y_test = torch.from_numpy(y_test).to(torch.float32)

In [263]:
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)

In [264]:
# -- MODEL: LSTM -- 

class Earn_LSTM(nn.Module):
    def __init__ (self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=13, hidden_size=15, num_layers=2, batch_first=True, dropout=0.3)
        self.linear = nn.Linear(in_features=15, out_features=1)

    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        return self.linear(last_out)
    

# -- METRICS -- 
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_true, y_pred).sum().item()
  acc = (correct / len(y_pred)) * 100
  return acc

# -- MODEL, LOSS AND OPTIMIZER

modelv2 = Earn_LSTM()

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adadelta(lr=0.18, params=modelv2.parameters())

In [265]:
# -- TRAINING LOOP -- 

SEED = 76
epochs = 70

torch.manual_seed(SEED)

for epoch in range(epochs):
    modelv2.train()
    for x_train, y_train in train_loader:
        y_logits = modelv2(x_train).squeeze(1)
        y_preds = torch.round(torch.sigmoid(y_logits))

        loss = loss_fn(y_logits, y_train)
        acc = accuracy_fn(y_true=y_train, y_pred=y_preds)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    modelv2.eval()
    with torch.inference_mode():
        for x_val, y_val in val_loader:
            val_logits = modelv2(x_val).squeeze(1)
            val_preds = torch.round(torch.sigmoid(val_logits))

            val_loss = loss_fn(val_logits, val_preds)
            val_acc = accuracy_fn(y_true=y_val, y_pred=val_preds)
    
    if epoch % 10 == 0:
        print(f"Epoch: {epoch} | Loss: {loss:.5f}, Accuracy: {acc:.2f}% | Val loss: {val_loss:.5f}, Val acc: {val_acc:.2f}%")

Epoch: 0 | Loss: 0.70184, Accuracy: 50.00% | Val loss: 0.65859, Val acc: 30.00%
Epoch: 10 | Loss: 0.59996, Accuracy: 78.57% | Val loss: 0.41374, Val acc: 80.00%
Epoch: 20 | Loss: 0.63661, Accuracy: 71.43% | Val loss: 0.32959, Val acc: 50.00%
Epoch: 30 | Loss: 0.42152, Accuracy: 92.86% | Val loss: 0.22940, Val acc: 60.00%
Epoch: 40 | Loss: 0.29651, Accuracy: 100.00% | Val loss: 0.33348, Val acc: 80.00%
Epoch: 50 | Loss: 0.33417, Accuracy: 85.71% | Val loss: 0.27309, Val acc: 60.00%
Epoch: 60 | Loss: 0.28814, Accuracy: 78.57% | Val loss: 0.29108, Val acc: 90.00%


In [267]:
# -- TEST -- 
modelv2.eval()
with torch.no_grad():
    for x, y in test_loader:
        test_logits = modelv2(x).squeeze(1)
        test_preds = torch.round(torch.sigmoid(test_logits))

        test_loss = loss_fn(test_logits, y)
        test_acc = accuracy_fn(y_true=y, y_pred=test_preds)

        print(f"Loss: {test_loss:.2f} | Accuracy {test_acc:.2f}%")

Loss: 0.68 | Accuracy 56.25%
Loss: 1.11 | Accuracy 56.25%
Loss: 1.01 | Accuracy 50.00%
Loss: 0.84 | Accuracy 68.75%
Loss: 1.21 | Accuracy 50.00%
Loss: 0.80 | Accuracy 62.50%
Loss: 0.73 | Accuracy 62.50%
Loss: 0.67 | Accuracy 68.75%
Loss: 0.79 | Accuracy 68.75%
Loss: 0.66 | Accuracy 75.00%
Loss: 1.04 | Accuracy 56.25%
Loss: 0.75 | Accuracy 68.75%
Loss: 0.86 | Accuracy 56.25%
Loss: 1.11 | Accuracy 50.00%
